# Notebook 1: Data loading and preprocessing.

This notebook walks through the complete preprocessing pipeline: 
Goal is preparing data to be explored correctly without bias or misleading results.

| Step | Operation | Justification |
|------|-----------|---------------|
| 1 | Drop NaN values | A few records were found containing NaN|
| 2 | Drop duplicates | Avoid biasing the model toward repeated flows |
| 3 | Replace Inf/-Inf with NaN | CICFlowMeter produces Inf when flow duration = 0 |
| 4 | Drop high-missing columns (>50%) | Columns with excessive missing data add noise |
| 5 | Removing inconsistenicies | ...|
| 6 | Removing noise | ... |

## Loading Libraries

In [25]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import glob
import os

## Merging all CSV files into single CSV file.

In [26]:
# Folder containing CSV files
folder_path = "C:\\Users\\abdul\\projects\\n_a_i_d\\data\\raw"

# Get all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

output_file = "merged.csv"

first = True

# Read and merge
for file in csv_files:
    print(f"Processing {file}...")
    
    for chunk in pd.read_csv(file, chunksize=50000):
        chunk.to_csv(# Save
            output_file,
            mode='w' if first else 'a',
            index=False,
            header=first
        )
        first = False

print("Merging complete!")

Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Friday-WorkingHours-Morning.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Monday-WorkingHours.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Tuesday-WorkingHours.pcap_ISCX.csv...
Processing C:\Users\abdul\projects\n_a_i_d\data\raw\Wednesday-workingHours.pcap_ISCX.csv...
Merging complete!


## Loading the dataset.

In [27]:
df = pd.read_csv("C:\\Users\\abdul\\projects\\n_a_i_d\\data\\processed\\merged.csv")
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2830743 entries, 0 to 2830742
Data columns (total 79 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0    Destination Port             int64  
 1    Flow Duration                int64  
 2    Total Fwd Packets            int64  
 3    Total Backward Packets       int64  
 4   Total Length of Fwd Packets   int64  
 5    Total Length of Bwd Packets  int64  
 6    Fwd Packet Length Max        int64  
 7    Fwd Packet Length Min        int64  
 8    Fwd Packet Length Mean       float64
 9    Fwd Packet Length Std        float64
 10  Bwd Packet Length Max         int64  
 11   Bwd Packet Length Min        int64  
 12   Bwd Packet Length Mean       float64
 13   Bwd Packet Length Std        float64
 14  Flow Bytes/s                  float64
 15   Flow Packets/s               float64
 16   Flow IAT Mean                float64
 17   Flow IAT Std                 float64
 18   Flow IAT Max                 int

In [29]:
df.columns = df.columns.str.strip()

In [30]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

In [31]:
df.isna().sum()

Destination Port               0
Flow Duration                  0
Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
                              ..
Idle Mean                      0
Idle Std                       0
Idle Max                       0
Idle Min                       0
Label                          0
Length: 79, dtype: int64

In [32]:
# df[df.duplicated()].sum()  
#DUPLICATES ARE NOT APPEARING PROPERLY. WILL BE CHECKED LATER .

In [33]:
attack_count = df["Label"].value_counts().drop("BENIGN").sum()
attack_count

np.int64(557646)

In [34]:
df.to_csv("processed_merged.csv" , index=False)